# 01 - Introducao ao Tool Use
> Como dar ferramentas ao Claude e processar chamadas

**Modulo:** `03_tool_use` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
tools = [{'name':'calcular',
    'description':'Calcula expressoes matematicas Python.',
    'input_schema':{'type':'object',
        'properties':{'expr':{'type':'string'}},'required':['expr']}}]

def calcular(expr):
    try: return str(eval(expr, {'__builtins__':{}}, {}))
    except Exception as e: return f'Erro: {e}'

def loop(pergunta):
    msgs=[{'role':'user','content':pergunta}]
    while True:
        r=client.messages.create(model=HAIKU,max_tokens=512,tools=tools,messages=msgs)
        if r.stop_reason=='end_turn': return r.content[0].text
        msgs.append({'role':'assistant','content':r.content})
        res=[]
        for b in r.content:
            if b.type=='tool_use':
                res.append({'type':'tool_result','tool_use_id':b.id,'content':calcular(**b.input)})
        msgs.append({'role':'user','content':res})

print(loop('Quanto e 15% de 840 mais 72 dividido por 8?'))

## Exercicios

### Exercicio 1 — Nova ferramenta: conversor de temperatura
Crie uma tool `converter_temp` que converte Celsius para Fahrenheit (e vice-versa).
Integre no loop agentico e teste com: *"Quanto e 38°C em Fahrenheit?"*

In [ ]:
# Exercicio 1 — Conversor de temperatura

tools_temp = [
    {
        'name': 'converter_temp',
        'description': 'Converte temperatura entre Celsius e Fahrenheit. Use direction="C_to_F" ou "F_to_C".',
        'input_schema': {
            'type': 'object',
            'properties': {
                'valor': {'type': 'number', 'description': 'Valor da temperatura'},
                'direction': {
                    'type': 'string',
                    'enum': ['C_to_F', 'F_to_C'],
                    'description': 'Direcao da conversao'
                }
            },
            'required': ['valor', 'direction']
        }
    }
]

def converter_temp(valor, direction):
    if direction == 'C_to_F':
        return str(valor * 9/5 + 32)
    elif direction == 'F_to_C':
        return str((valor - 32) * 5/9)
    return 'Erro: direction deve ser C_to_F ou F_to_C'

def loop_temp(pergunta):
    msgs = [{'role': 'user', 'content': pergunta}]
    while True:
        r = client.messages.create(model=HAIKU, max_tokens=512, tools=tools_temp, messages=msgs)
        if r.stop_reason == 'end_turn':
            return r.content[0].text
        msgs.append({'role': 'assistant', 'content': r.content})
        res = []
        for b in r.content:
            if b.type == 'tool_use':
                resultado = converter_temp(**b.input)
                res.append({'type': 'tool_result', 'tool_use_id': b.id, 'content': resultado})
        msgs.append({'role': 'user', 'content': res})

# Teste
print(loop_temp('Quanto e 38°C em Fahrenheit?'))
print()
print(loop_temp('Converte 100°F pra Celsius'))

### Exercicio 2 — Combinando tools: calculadora + conversor
Monte um loop que disponibiliza DUAS tools ao mesmo tempo (`calcular` e `converter_temp`).
Teste com: *"Quanto e 20% de 180 graus Celsius em Fahrenheit?"*

In [ ]:
# Exercicio 2 — Duas tools no mesmo loop

tools_combo = [
    {
        'name': 'calcular',
        'description': 'Calcula expressoes matematicas Python.',
        'input_schema': {
            'type': 'object',
            'properties': {'expr': {'type': 'string'}},
            'required': ['expr']
        }
    },
    {
        'name': 'converter_temp',
        'description': 'Converte temperatura entre Celsius e Fahrenheit. Use direction="C_to_F" ou "F_to_C".',
        'input_schema': {
            'type': 'object',
            'properties': {
                'valor': {'type': 'number', 'description': 'Valor da temperatura'},
                'direction': {
                    'type': 'string',
                    'enum': ['C_to_F', 'F_to_C'],
                    'description': 'Direcao da conversao'
                }
            },
            'required': ['valor', 'direction']
        }
    }
]

# Dispatcher: mapeia nome da tool -> funcao Python
dispatch = {
    'calcular': lambda **kw: calcular(**kw),
    'converter_temp': lambda **kw: converter_temp(**kw),
}

def loop_combo(pergunta):
    msgs = [{'role': 'user', 'content': pergunta}]
    while True:
        r = client.messages.create(model=HAIKU, max_tokens=512, tools=tools_combo, messages=msgs)
        if r.stop_reason == 'end_turn':
            return r.content[0].text
        msgs.append({'role': 'assistant', 'content': r.content})
        res = []
        for b in r.content:
            if b.type == 'tool_use':
                fn = dispatch.get(b.name)
                resultado = fn(**b.input) if fn else f'Erro: tool {b.name} nao encontrada'
                print(f'  🔧 {b.name}({b.input}) → {resultado}')
                res.append({'type': 'tool_result', 'tool_use_id': b.id, 'content': resultado})
        msgs.append({'role': 'user', 'content': res})

# Teste — requer as duas tools em sequencia
print(loop_combo('Quanto e 20% de 180 graus Celsius em Fahrenheit?'))

### Exercicio 3 — Tool com validacao: buscador de CEP
Crie uma tool `buscar_cep` que recebe um CEP (string de 8 digitos) e retorna dados ficticios de endereco.
A tool deve **validar** se o CEP tem 8 digitos numericos antes de processar.
Teste com CEPs validos e invalidos.

## Proximos passos
- Proximo notebook do modulo
- [docs.anthropic.com](https://docs.anthropic.com)
